# Notebook 05 — Conclusiones
## Proyecto Integrador | Minería de Datos I
**Integrante:** Val Martinetti

---
Este notebook consolida los hallazgos de los notebooks anteriores, los diferencia según su naturaleza (evidencia / interpretación / conclusión) y propone próximos pasos concretos.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/streaming_users_clean.csv')
log = pd.read_csv('../logs/pipeline_log.csv')

print(f"Dataset limpio: {df.shape[0]} usuarios × {df.shape[1]} variables")
print(f"Retención del pipeline: {log['Retención (%)'].iloc[-1]}%")

## 1. Hallazgos por etapa

### 1.1 Inspección y calidad del dataset

In [ ]:
print("""
EVIDENCIA (observable en los datos):
  - Dataset original: 8.160 registros × 8 columnas
  - 126 duplicados exactos (sin considerar user_id)
  - Variantes en categóricas: 15 en subscription_plan, 26 en country, 28 en favorite_genre
  - 120 valores imposibles en age (< 6 o > 100 años)
  - 49 valores negativos en monthly_watch_time_mins
  - 29 valores negativos en customer_support_tickets
  - 139 valores extremos en watch_time winsorizados (k=3, límite 2.705 min)
  - 81 outliers en tickets winsorizados (k=3, límite 4 tickets)
  - 769 fechas no utilizables en last_login_date (320 nulas + 449 inválidas)

INTERPRETACIÓN:
  - Los problemas de codificación en categóricas (mayúsculas, tildes, abreviaturas, 
    idiomas) sugieren datos provenientes de múltiples sistemas de ingesta sin 
    esquema de validación unificado.
  - Los valores 99, 150 y 99.999 son 'valores centinela': indicadores de error en 
    la carga, comunes en sistemas legacy donde el campo no admite NULL.
  - Las fechas inválidas (mes 15) confirman falta de validación en el sistema origen.
  - El mecanismo de falta en watch_time es MAR: la tasa de faltantes en Premium 
    (10.4%) es 8× la de Básico (1.3%), lo que sugiere que la pérdida de datos está 
    asociada al plan, no al azar puro.

CONCLUSIÓN:
  El dataset tenía problemas de calidad sistemáticos y heterogéneos, propios de un 
  sistema con múltiples fuentes de ingesta sin validación centralizada. El pipeline 
  de limpieza resolvió todos los problemas con criterio de dominio, alcanzando 
  98.46% de retención estructural: se perdió 1.54% solo por duplicados verificados.
""")

### 1.2 EDA

In [ ]:
print("""
EVIDENCIA:
  - Distribución de planes: Básico 44.9% | Estándar 35.3% | Premium 19.8%
  - Edad: media 33 años, distribución casi simétrica (skew ≈ 0.14)
  - Watch time por plan: Básico ~540 min | Estándar ~750 min | Premium ~1.300 min/mes
  - Correlación edad–watch_time: r ≈ 0.00
  - Correlaciones entre las 3 variables numéricas: todas < 0.05 en valor absoluto
  - Distribución de géneros y países: uniforme (sin dominancia marcada)

INTERPRETACIÓN:
  - La brecha de consumo entre Premium y Básico (+140%) no puede explicarse solo 
    por diferencia de acceso a contenido: los heavy users eligen Premium, pero 
    también es posible que el acceso a más contenido genere mayor consumo.
  - La independencia de la edad respecto al consumo es un hallazgo contraintuitivo: 
    sugiere que la plataforma atrae y retiene usuarios de todas las edades con 
    patrones de uso similares, más allá de los estereotipos generacionales.
  - La distribución casi uniforme entre países refleja una presencia regional 
    equilibrada; ningún mercado domina.

CONCLUSIÓN:
  El plan de suscripción es el principal diferenciador del comportamiento de uso 
  de la plataforma. La edad, el género favorito y el país no predicen el nivel de 
  consumo. Para una segmentación accionable, el plan es el eje más relevante.
""")

### 1.3 PCA

In [ ]:
print("""
EVIDENCIA:
  - Variables de entrada: age, monthly_watch_time_mins, customer_support_tickets
  - Escalamiento: StandardScaler (obligatorio; varianzas heterogéneas)
  - PC1: ~34% varianza | PC2: ~33% | PC3: ~33%
  - Carga dominante en PC1: monthly_watch_time_mins
  - Carga dominante en PC2: age + customer_support_tickets (combinación)
  - 2 componentes explican ~67% de la varianza

INTERPRETACIÓN:
  - La distribución casi uniforme de varianza entre las 3 PCs es consecuencia 
    directa de la baja correlación entre variables (verificada en EDA). 
    Cuando las variables son ortogonales, PCA no puede comprimir: cada PC 
    hereda aproximadamente una variable.
  - PC1 actúa como 'eje de consumo': separa heavy users de light users.
  - PC2 captura una combinación de edad y demanda de soporte: usuarios mayores 
    con más tickets se ubican en extremos positivos.

CONCLUSIÓN:
  El PCA confirma que las variables cuantitativas del dataset capturan dimensiones 
  de variación independientes entre sí. La baja compresibilidad es informativa: 
  no hay redundancia en las métricas disponibles. Para un PCA más potente, 
  sería necesario incorporar variables adicionales o transformadas (e.g., 
  plan_encoded, días_desde_last_login).
""")

## 2. Limitaciones del análisis

In [ ]:
print("""
LIMITACIONES IDENTIFICADAS:
══════════════════════════════════════════════════════════════

1. MECANISMO MNAR NO RESUELTO EN FECHAS:
   - 769 fechas de último login son NaT. Sin estas fechas, no es posible 
     analizar la actividad reciente ni calcular métricas de retención/churn.
   - Limitación: no se puede caracterizar a usuarios inactivos vs. activos.

2. VARIABLES CATEGÓRICAS EXCLUIDAS DEL PCA:
   - subscription_plan, country y favorite_genre son altamente relevantes 
     para el negocio pero no se incorporaron al PCA por ser nominales. 
   - Una codificación dummy o target encoding las habilitaría en futuros análisis.

3. CAUSALIDAD NO ESTABLECIDA:
   - La relación plan–watch_time es una asociación observada, no una causa.
   - No es posible determinar si los heavy users eligen Premium o si Premium 
     genera mayor consumo.

4. AUSENCIA DE DATOS HISTÓRICOS:
   - El dataset es un snapshot. Sin series temporales no es posible analizar 
     tendencias de crecimiento, churn o estacionalidad.

5. TICKETS DE SOPORTE POST-WINSORIZACIÓN:
   - Se acotaron a máximo 4. Los casos extremos (99, 150) perdieron su 
     magnitud real, aunque representaban errores de carga según el análisis.
""")

## 3. Próximos pasos

In [ ]:
print("""
PRÓXIMOS PASOS RECOMENDADOS:
══════════════════════════════════════════════════════════════

INMEDIATO (con los datos actuales):
  1. Codificar subscription_plan (ordinal: Básico=1, Estándar=2, Premium=3) 
     e incorporarla al PCA para obtener una separación más clara.
  2. Calcular días_desde_last_login (para los registros con fecha válida) 
     como proxy de actividad reciente.
  3. Calcular watch_time_por_hora_dia como normalización (si hubiera fechas).

MEJORAS EN LA FUENTE DE DATOS:
  4. Implementar validación de esquema en la ingesta: rangos válidos, 
     valores posibles para categóricas, formato de fecha estricto.
  5. Unificar la codificación de categóricas en el sistema origen 
     (usar IDs de país/plan en lugar de strings libres).

MODELADO FUTURO:
  6. Clustering (K-Means) sobre los scores PCA para segmentación de usuarios.
  7. Modelo de predicción de churn usando plan, watch_time y tickets como features.
  8. Análisis de cohortes si se obtienen datos históricos.

COMUNICACIÓN:
  9. Compartir el análisis con el equipo de producto: la brecha Premium–Básico 
     en consumo (+140%) es un input directo para decisiones de pricing.
  10. Investigar cualitativamente por qué watch_time en Premium tiene mayor 
      tasa de nulos (posible problema técnico de tracking en ese plan).
""")

## 4. Tabla resumen de evidencia → interpretación → conclusión

In [ ]:
resumen = pd.DataFrame({
    'Hallazgo': [
        'Pipeline retención 98.46%',
        'Premium: 10× más faltantes en watch_time',
        'Premium: +140% watch_time vs Básico',
        'Correlación edad–watch_time ≈ 0',
        'PCA: varianza ~33% por componente',
    ],
    'Tipo': [
        'Evidencia',
        'Evidencia + Interpretación (MAR)',
        'Evidencia + Conclusión',
        'Evidencia',
        'Evidencia + Interpretación',
    ],
    'Implicancia': [
        'Calidad del proceso de limpieza',
        'Posible bug de tracking en cuentas Premium',
        'Plan es el principal segmentador de uso',
        'Edad no es predictor de consumo',
        'Variables numéricas independientes entre sí',
    ]
})
print(resumen.to_string(index=False))